In [8]:
import yfinance as yf
import numpy as np
import pandas as pd
from pathlib import Path
from config import RAW_DATA_DIR

def _download_returns(serie_id, start, end=None, freq='ME'):
    """Downloads historical price data and calculates log returns."""
    data = yf.download(serie_id, start=start, end=end, auto_adjust=True)
    
    if data.empty:
        raise ValueError(f"yfinance returned no data for ticker: {serie_id}")
        
    price = data['Close']
    price_m = price.resample(freq).last().dropna()
    
    # Calculate log returns
    returns = np.log(price_m).diff().dropna()
    
    return returns

def _save_returns(out_path, returns, series_name):
    """Converts a Series to a DataFrame, ensures directories exist, and saves to CSV."""
    returns.columns = [series_name]
    out_path.parent.mkdir(parents=True, exist_ok=True)
    returns.to_csv(out_path)

def process_series_batch(specs):
    """
    Takes a list of dictionaries containing download specifications, applies 
    defaults, infers paths if necessary, and saves the returns.
    """
    for spec in specs:
        # 1. Extract parameters and apply defaults
        serie_id = spec.get('serie_id')
        if not serie_id:
            print("Skipping entry: Missing required 'serie_id'.")
            continue
            
        start = spec.get('start')
        if not start:
            print(f"Skipping {serie_id}: Missing required 'start' date.")
            continue
            
        end = spec.get('end', None)          # Default to None (today)
        freq = spec.get('freq', 'ME')        # Default to Month End
        name = spec.get('name', serie_id)    # Default to ticker if no name provided
        
        # 2. Path inference
        custom_out_path = spec.get('out_path')
        if custom_out_path:
            # If user provided a specific relative path
            final_out_path = RAW_DATA_DIR / custom_out_path
        else:
            # Infer path: RAW_DATA_DIR / Name / freq.csv
            final_out_path = RAW_DATA_DIR / f"returns_{name}_{freq}.csv"
            
        # 3. Download and Save
        print(f"Processing {name} ({serie_id}) at {freq} frequency...")
        try:
            returns = _download_returns(serie_id, start=start, end=end, freq=freq)
            _save_returns(final_out_path, returns, name)
            print(f"  -> Successfully saved to {final_out_path}")
        except Exception as e:
            print(f"  -> Failed to process {name}: {e}")

In [9]:
my_specifications = [
        {
            'serie_id': "^GSPC", 
            'start': "1950-01-01", 
            'freq': 'ME', 
            'name': 'SP500', 
            #'out_path': 'Indices/SP500_Monthly.csv'
        },
        {
            'serie_id': 'AAPL', 
            'start': '2000-01-01',
            'freq': 'ME', 
        },
    ]
    
process_series_batch(my_specifications)

Processing SP500 (^GSPC) at ME frequency...


[*********************100%***********************]  1 of 1 completed


  -> Successfully saved to C:\Users\Fabrizio Ortega\Documents\TFM\sharpe_ratio_TFM\resources\data\raw\returns_SP500_ME.csv
Processing AAPL (AAPL) at ME frequency...


[*********************100%***********************]  1 of 1 completed

  -> Successfully saved to C:\Users\Fabrizio Ortega\Documents\TFM\sharpe_ratio_TFM\resources\data\raw\returns_AAPL_ME.csv
